# Vocabulary expansion on SmolLM2-135M → Greek (Colab driver)

Runs the whole study on a **free Colab T4**: data preparation → tokenizer extension + QA →
throughput calibration → baseline / Exp1 / Exp2 with **matched tokens and steps** → evaluation
→ results table and plots. Everything is cached on Google Drive and every run is resumable,
so a disconnected session continues where it stopped.

| stage | time on a T4 (approx.) |
|---|---|
| data prep (stream ~230 MB, clean, dedup, split) | 10–20 min, once |
| continued-BPE tokenizer (60 MB) + QA | ~3 min, once |
| baseline eval | ~7 min |
| Exp1 / Exp2 training (each) | ≤ 30 min + dev evals |
| final eval per run | ~7 min (Exp2 also evaluates its step-0 state) |

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code + persistent storage
Either clone the repository (set `REPO_URL`) or upload the repo folder to Drive as `MyDrive/vocabx/repo`.
Artifacts (data, tokenizers, checkpoints, results) go to `MyDrive/vocabx/artifacts`.

In [ ]:
REPO_URL = "https://github.com/nevasini1/smollm2-greek-vocab-expansion.git"   # set to "" to use a copy uploaded to Drive instead
DRIVE_ROOT = "/content/drive/MyDrive/vocabx"

from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
if REPO_URL:
    !rm -rf /content/vocabx && git clone -q {REPO_URL} /content/vocabx
else:
    !rm -rf /content/vocabx && cp -r {DRIVE_ROOT}/repo /content/vocabx
%cd /content/vocabx
!pip install -q -e .
WORK = f"{DRIVE_ROOT}/artifacts"
W = f"--set workdir={WORK}"
print("artifacts ->", WORK)

In [ ]:
# Optional: a Hugging Face token raises Hub rate limits (all default datasets are ungated).
# from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 2. Data: training pools + fixed, decontaminated eval sets
Greek: Wikipedia (el) + FineWeb-2 ell_Grek (75 MB each) for training; held-out Wikipedia docs, the
official FineWeb-2 test split and Belebele/FLORES passages for evaluation. English: FineWeb-Edu
(SmolLM2's own distribution; also the replay pool), BBC News Jan–Jun 2025 (after SmolLM2's release)
and the parallel Belebele passages. Held-out docs are drawn first; training docs that share a
paragraph or are MinHash near-duplicates of any held-out doc are dropped.

In [ ]:
!python -m vocabx prepare --config configs/exp2_extend.yaml {W}

## 3. Extend the tokenizer (continued BPE, +8,000 Greek tokens) and run the QA harness
Checks: original ids/merges untouched · English and any non-Greek text tokenize identically ·
`decode(encode(x)) == x` · every new token reachable (self-tokenization test) · compression and
vocab-size curve (1k…16k from merge prefixes).

In [ ]:
!python -m vocabx tokenizer --config configs/exp2_extend.yaml {W} | head -80

## 4. Fix the matched step budget ONCE (calibrated on the slower arm, Exp2)
`calibrate` measures throughput on this GPU and writes `artifacts/budget.json` with
`max_steps = min(steps that fit 27 min, steps the Greek pool covers without repetition)`.
Every later run, in this and any future session, reads that file, so a reconnect can never
change the budget between arms. Delete `budget.json` (or pass `--force`) only to start over.

In [ ]:
!python -m vocabx calibrate --config configs/exp2_extend.yaml {W} --steps 20 --minutes 27
import json
budget = json.load(open(f"{WORK}/budget.json"))
cal = budget["from_calibration"]
print({k: cal[k] for k in ["tokens_per_second", "suggested_max_steps", "pool_cap_steps", "recommended_max_steps",
                           "peak_memory_gb", "greek_MB_available", "sdpa"]})
print("matched budget: max_steps =", budget["max_steps"], f"({budget['max_steps'] * cal['tokens_per_step'] / 1e6:.1f}M tokens per arm)")

## 5. (Optional) LR probe on dev data, both arms, 150 steps each (~1 h total)
Pre-registered rule: the largest LR whose English-dev loss is ≤ 1/3 of its Greek-dev gain, both
measured from the ORIGINAL model and averaged over arms. The choice is written to
`artifacts/lr_choice.json`, and all later trained runs pick it up automatically. Run it before §6.

In [ ]:
RUN_PROBE = False
if RUN_PROBE:
    !python -m vocabx plan configs/plans/lr_probe.yaml {W}
    !python -m vocabx curves --workdir {WORK} --runs probe_exp1_lr1e-4 probe_exp2_lr1e-4 probe_exp1_lr3e-4 probe_exp2_lr3e-4 probe_exp1_lr1e-3 probe_exp2_lr1e-3

## 6. Main comparison: baseline, Exp1, Exp2 (matched tokens and steps)
Resumable: if Colab disconnects, re-run this cell; finished runs are skipped and the
interrupted one resumes from its last checkpoint.

In [ ]:
!python -m vocabx plan configs/plans/main.yaml {W}

In [ ]:
from IPython.display import Markdown, Image, display
R = f"{WORK}/results/main"
display(Markdown(open(f"{R}/results.md").read()))
for img in ["learning_curves.png", "tradeoff_trajectories.png", "english_mass_on_new_tokens.png"]:
    if os.path.exists(f"{R}/{img}"):
        display(Image(f"{R}/{img}"))

## 6b. (Optional) Weight-interpolation frontiers (eval only, ~10 min per run on dev sets)
θ(α) = (1−α)·θ_start + α·θ_final for each arm. θ_start is the expanded step-0 model for Exp2. Comparing frontiers shows English given up per unit of Greek gained, independent of where the 30-min run stopped.

In [ ]:
DEV = "wikipedia_el:dev fineweb2_el:dev fineweb_edu_en:dev"
!python -m vocabx interpolate --config configs/exp1_finetune.yaml {W} --sets {DEV}
!python -m vocabx interpolate --config configs/exp2_extend.yaml {W} --sets {DEV} --plot-runs exp1_finetune exp2_extend
display(Image(f"{WORK}/results/interpolation_frontiers.png"))

## 7. Ablations and seeds (optional; each ≈ one more training run)
Run a prefix of the ranked list, e.g. the byte-matched Exp2 and the 20% English-replay pair.

In [ ]:
!python -m vocabx plan configs/plans/ablations.yaml {W} --only exp2_bytes_matched exp1_replay20 exp2_replay20

In [ ]:
# Training-noise estimate (two more seeds per arm, common-random-number pairs):
# !python -m vocabx plan configs/plans/seeds.yaml {W}

In [ ]:
R = f"{WORK}/results/ablations"
if os.path.exists(f"{R}/results.md"):
    display(Markdown(open(f"{R}/results.md").read()))